# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neilivanorencia/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

A model is only worth something if it beats a rule a person can read. This notebook builds that rule: three candidate rules are compared on the three development pairs, the hardest one to beat is frozen by a criterion fixed in advance, and the frozen rule ranks every page of April 2026 into a review queue with an action and reason codes. The top 20 are then reviewed one by one against what actually happened in May.

**How to read it:** each section opens with the thinking, then a **Method** note explaining what the next code cell computes, then the output, then a **Finding** note on what it shows. Terms from the earlier notebooks (month M, month M+1, expected clicks, the target, Precision@K, Capture@50, size bands) still apply; the new ones are defined below.


## Key terms

| Term                | Technical meaning                                                                          | In plain words                                                       |
| ------------------- | ------------------------------------------------------------------------------------------ | -------------------------------------------------------------------- |
| **Candidate rule**  | A hand-written scoring formula with no fitted weights                                      | A recipe for ranking pages that anyone can read and check            |
| **Frozen baseline** | The candidate chosen by a criterion fixed before the results, then never changed           | The fixed bar every later model must clear                           |
| **Review queue**    | All pages with a real shortfall in month M, ranked by the frozen rule                      | The editor's to-do list, most important first                        |
| **Reason code**     | A tag recording why a page scored, or what else is true of it                              | The "why" next to each page, so the list can be trusted              |
| **Action**          | One recommended next step per page, chosen from its reason codes in a fixed priority order | What the editor should do first                                      |
| **Confidence note** | High, medium, or low, from how certain the shortfall is and how stable the page is         | How much weight to give the pick                                     |
| **Client factor**   | A client's total clicks ÷ its total expected clicks from position alone (+1 on both sides) | Whether a client's whole site clicks above or below normal           |
| **Engaged share**   | GA4 engaged sessions ÷ GA4 sessions                                                        | The share of visits where the visitor actually stayed and interacted |


**Method:** this cell makes sure DuckDB is version 1.1 or newer, the first version whose Hugging Face login can read the token from the environment.


In [1]:
%pip -q install "duckdb>=1.1"

## Setup

**Method, step by step:**

1. **Finds the project folder** and **reads the Hugging Face token**, as in the data contract: environment variable or `.env` file, then a Colab Secret named `HF_TOKEN`, then a typed prompt as the last resort. DuckDB reads the token from the environment, so it never appears in a query or an output.
2. **Loads the tools.** `duckdb` queries the warehouse, `pandas` and `numpy` handle the results, and `scipy` supplies the two luck tests (`poisson` for rule A, the negative binomial for rule B and the target).
3. **Defines the months and the cache.** The same three development pairs as the data contract; June 2026 is never read. Heavy queries go through `cached()`, which saves each result once under `work/outputs/w03_cache/` and reuses it on later runs.
4. **Defines `show()` and `show_table()`**, which display every result as a titled table.


In [2]:
import getpass
import os
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import nbinom, poisson

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

**Connected to the warehouse**

| Measure | Value |
| --- | --- |
| token read from | Colab Secret |
| token shown anywhere | no (DuckDB reads it from the environment) |
| development pairs | 2026-02 → 2026-03, 2026-03 → 2026-04, 2026-04 → 2026-05 |
| sealed pair, never read | 2026-05 → 2026-06 |
| cache folder | work/outputs/w03_cache |

## Data

The rules are judged on the same three development pairs as the data contract, rebuilt with its rules. June 2026 stays sealed.


**Method:** this cell builds the page-month table for February to May 2026 (one row per page per month, position pieces only from days with a valid position, GA4 numbers only where `ga4_data_available IS TRUE`), now including GA4 engaged sessions for the engagement reason code. Each month is scanned once and cached. It also reads each client's search-history start date and each page's content type, search intent, and latest optimization date.


In [3]:
# One row per page per month, with GA4 engaged sessions added for the engagement reason code.
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)
assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
content = con.sql(f"SELECT content_hash_id, content_type, main_intent, last_optimized_date FROM {DIM_CONTENT}").df()
show_table("Inputs loaded", pd.DataFrame({
    "Rows": [f"{len(page_months):,}", f"{len(clients):,}", f"{len(content):,}"],
    "What it is": ["Page-months, February to May 2026", "Clients with their search-history start date",
                   "Pages with content type, search intent, and latest optimization date"],
}, index=pd.Index(["page_months", "clients", "content"], name="Table")))

**Inputs loaded**

| Table | Rows | What it is |
| --- | ---: | --- |
| page_months | 762,967 | Page-months, February to May 2026 |
| clients | 104 | Clients with their search-history start date |
| content | 519,606 | Pages with content type, search intent, and latest optimization date |

**Method:** this cell applies the data contract's rules to each development pair (impressions in both months, client history covering all of month M, a valid position in both months, not optimized during the pair). For each month it computes the impression-weighted position, its daily spread, expected clicks from the fine position bins and from the five tiers (both client-scaled), and whether the page fell short by the locked rule: clicks at most half of expected, with negative binomial p below 0.05. The spread factor α is estimated exactly as the data contract does (one estimate per pair from pages whose position barely moved, the median rounded to three decimals), so every notebook shares one target. The month M+1 version of the rule is the target.


In [4]:
# Rules and bins exactly as in the data contract; the spread factor is estimated below with its method.
BIN_EDGES = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 20.5, 50.5]
BIN_NAMES = [str(position) for position in range(1, 11)] + ["11-20", "21-50", "50+"]


def position_bin(position: pd.Series) -> pd.Series:
    """Return each page's position bin: exact rank up to 10, then 11-20, 21-50, and 50+."""
    return pd.Series(np.array(BIN_NAMES)[np.searchsorted(BIN_EDGES, position, side="right")], index=position.index)


def position_tier(position: pd.Series) -> pd.Series:
    """Return the five tiers used by the earlier score: below 3, below 10, below 20, below 50, and 50 or beyond."""
    tiers = np.select([position < 3, position < 10, position < 20, position < 50], ["top_3", "page_1", "striking", "page_3_5"], "deep")
    return pd.Series(tiers, index=position.index)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str, group: pd.Series) -> pd.Series:
    """Return expected clicks from each group's pooled click rate, scaled by the page's client factor (+1 on both sides)."""
    rate = frame[clicks].groupby(group).sum() / frame[impressions].groupby(group).sum()
    expected = frame[impressions] * group.map(rate)
    by_client = frame.assign(_expected=expected).groupby("client_hash_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_hash_id"].map(factor)


def covered_clients(month_m: str) -> pd.Series:
    """Return the clients whose search history starts on or before the first day of month M."""
    return clients.loc[clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"), "client_hash_id"]


def add_month_measures(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add position, its spread, click-through, and both expectations for one month."""
    impressions, clicks = rows[f"impressions_{suffix}"], rows[f"clicks_{suffix}"]
    rows[f"position_{suffix}"] = rows[f"sum_position_{suffix}"] / rows[f"position_impressions_{suffix}"]
    spread = rows[f"sum_position_sq_{suffix}"] / rows[f"position_impressions_{suffix}"] - rows[f"position_{suffix}"] ** 2
    rows[f"position_volatility_{suffix}"] = np.sqrt(spread.clip(lower=0))
    rows[f"ctr_{suffix}"] = clicks / impressions
    rows[f"expected_bins_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_bin(rows[f"position_{suffix}"]))
    rows[f"expected_tier_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_tier(rows[f"position_{suffix}"]))
    return rows


def add_luck_test(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add the negative binomial luck test and the locked shortfall rule for one month."""
    clicks, expected = rows[f"clicks_{suffix}"], rows[f"expected_bins_{suffix}"]
    rows[f"p_{suffix}"] = nbinom.cdf(clicks, 1 / ALPHA, 1 / (1 + ALPHA * expected))
    rows[f"shortfall_{suffix}"] = (rows[f"p_{suffix}"] < 0.05) & (clicks <= 0.5 * expected)
    return rows


def build_pair(month_m: str, month_next: str) -> pd.DataFrame:
    """Return one month pair's eligible rows with both months' measures and the locked target."""
    keys = ["client_hash_id", "content_hash_id"]
    first = page_months[page_months["month"] == month_m].set_index(keys)
    second = page_months[page_months["month"] == month_next].set_index(keys)
    rows = first.join(second, lsuffix="_m", rsuffix="_next", how="inner").reset_index()
    start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
    rows = rows[rows["client_hash_id"].isin(covered_clients(month_m))]
    rows = rows[(rows["position_impressions_m"] > 0) & (rows["position_impressions_next"] > 0)]
    rows = rows.merge(content, on="content_hash_id", how="left")
    rows = rows[~pd.to_datetime(rows["last_optimized_date"]).between(start, end)].copy()
    rows = add_month_measures(add_month_measures(rows, "m"), "next")
    rows["pair_end"] = end
    return rows.reset_index(drop=True)


def stable_pages(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Return pages whose position moved less than one place, and each one's expected share of clicks in month M+1.

    The expected share corrects for the overall change in click rate between the two months.
    """
    stable = frame[(frame["position_next"] - frame["position_m"]).abs() < 1]
    shift = (stable["clicks_next"].sum() / stable["impressions_next"].sum()) / (stable["clicks_m"].sum() / stable["impressions_m"].sum())
    share = shift * stable["impressions_next"] / (stable["impressions_m"] + shift * stable["impressions_next"])
    return stable, share



def spread_factor(frame: pd.DataFrame) -> float:
    """Return the method-of-moments estimate of the negative binomial spread factor alpha.

    For pages with a stable position, the gap between month-M+1 clicks and the clicks their month-M rate
    predicts has variance (Poisson part) + alpha x (squared clicks). Pooling the excess over the Poisson
    part across pages gives alpha.
    """
    stable, share = stable_pages(frame)
    scale = share / (1 - share)
    gap = stable["clicks_next"] - scale * stable["clicks_m"]
    excess = (gap ** 2 - stable["clicks_next"] - scale ** 2 * stable["clicks_m"]).sum()
    squared = ((stable["clicks_next"] ** 2 - stable["clicks_next"]) + scale ** 2 * (stable["clicks_m"] ** 2 - stable["clicks_m"])).sum()
    return float(excess / squared)



pairs = {f"{month_m} → {month_next}": build_pair(month_m, month_next) for month_m, month_next in PAIRS}
ALPHA = round(float(np.median([spread_factor(rows) for rows in pairs.values()])), 3)
for rows in pairs.values():
    add_luck_test(add_luck_test(rows, "m"), "next")
    rows["target"] = rows["shortfall_next"]
    rows["missed_next"] = (rows["expected_bins_next"] - rows["clicks_next"]).clip(lower=0).where(rows["target"], 0)
show_table(f"Development pairs with the locked target (α = {ALPHA:.3f})", pd.DataFrame({
    "Rows": [f"{len(rows):,}" for rows in pairs.values()],
    "Positives": [f"{rows['target'].sum():,}" for rows in pairs.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows in pairs.values()],
}, index=pd.Index(list(pairs), name="Pair")))

**Development pairs with the locked target (α = 0.039)**

| Pair | Rows | Positives | Base rate |
| --- | ---: | ---: | ---: |
| 2026-02 → 2026-03 | 114,972 | 8,465 | 7.4% |
| 2026-03 → 2026-04 | 155,707 | 10,031 | 6.4% |
| 2026-04 → 2026-05 | 157,530 | 6,525 | 4.1% |

**Finding:** the three pairs hold 114,972, 155,707, and 157,530 rows with 8,465, 10,031, and 6,525 real shortfalls (base rates 7.4%, 6.4%, and 4.1%), and the spread factor comes out at 0.039: the same rows and target as the data contract.


## 1. My rule and its reason codes

**The rule in plain words:** _rank pages by how many clicks they missed last month compared with pages at the same position on the same client's site, counting a page only when the gap is too large to be bad luck._ Three versions compete:

| Rule  | In plain words                                                                                                                                                                            |
| ----- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **A** | Missed clicks against the five-tier expectation, counted when Poisson says luck explains the gap less than 1% of the time (the score from the earlier framing)                            |
| **B** | Missed clicks against the finer position-bin expectation, counted when the page fell short by the target's own rule in month M (at most half of expected, negative binomial p below 0.05) |
| **C** | The stand-in CTR flag (500+ impressions, position 20 or better, click-through below 0.5%), largest pages first                                                                            |

**Reason codes and actions.** Every page in the queue carries `real_shortfall`, plus any of four more codes. The action follows the first code that applies, in this order:

| Reason code           | When it applies                                                           | Action                              |
| --------------------- | ------------------------------------------------------------------------- | ----------------------------------- |
| `rank_unstable`       | The daily position spread in month M is 5 places or more                  | Wait and re-check next month        |
| `client_wide_low_ctr` | The client's whole site earns under half the clicks its positions predict | Review site-wide                    |
| `commercial_intent`   | The keyword's intent is commercial or transactional                       | Check the search results page first |
| `low_engagement`      | 30 or more GA4 sessions, under half of them engaged                       | Improve the content                 |
| `real_shortfall` only | None of the above                                                         | Rewrite the title and description   |

**Method:** this cell writes the three rules, the reason codes, and the actions as code, with every threshold named once, and lists the inputs each rule reads.


In [5]:
# Every threshold named once.
FLAG_CUTOFF, FLAG_MIN_IMPRESSIONS = 0.005, 500
UNSTABLE_SPREAD, MIN_SESSIONS, ENGAGED_SHARE, CLIENT_LOW = 5, 30, 0.5, 0.5
BUYING = {"commercial", "transactional"}


def rule_a(rows: pd.DataFrame) -> pd.Series:
    """Missed clicks against the five-tier expectation, kept where Poisson luck explains the gap less than 1% of the time."""
    missed = (rows["expected_tier_m"] - rows["clicks_m"]).clip(lower=0)
    return missed.where(poisson.cdf(rows["clicks_m"], rows["expected_tier_m"]) < 0.01, 0)


def rule_b(rows: pd.DataFrame) -> pd.Series:
    """Missed clicks against the position-bin expectation, kept where the page fell short by the target's rule in month M."""
    missed = (rows["expected_bins_m"] - rows["clicks_m"]).clip(lower=0)
    return missed.where(rows["shortfall_m"], 0)


def rule_c(rows: pd.DataFrame) -> pd.Series:
    """The stand-in CTR flag, scored by impressions so the largest flagged pages come first."""
    flagged = (rows["impressions_m"] >= FLAG_MIN_IMPRESSIONS) & (rows["position_m"] < 20.5) & (rows["ctr_m"] < FLAG_CUTOFF)
    return rows["impressions_m"].where(flagged, 0).astype(float)


RULES = {"A": rule_a, "B": rule_b, "C": rule_c}
RULE_INPUTS = {
    "A": ["clicks_m", "impressions_m", "position_m", "client_hash_id"],
    "B": ["clicks_m", "impressions_m", "position_m", "client_hash_id"],
    "C": ["clicks_m", "impressions_m", "position_m"],
}
CODE_INPUTS = ["position_volatility_m", "clicks_m", "impressions_m", "position_m", "main_intent", "ga4_sessions_m", "ga4_engaged_sessions_m"]


def client_factor(rows: pd.DataFrame) -> pd.Series:
    """Return each page's client factor: the client's clicks ÷ its expected clicks from position bins alone, +1 on both sides."""
    group = position_bin(rows["position_m"])
    raw = rows["impressions_m"] * group.map(rows["clicks_m"].groupby(group).sum() / rows["impressions_m"].groupby(group).sum())
    by_client = rows.assign(_raw=raw).groupby("client_hash_id")
    return rows["client_hash_id"].map((by_client["clicks_m"].sum() + 1) / (by_client["_raw"].sum() + 1))


def reason_codes(rows: pd.DataFrame, score: pd.Series) -> pd.DataFrame:
    """Return one True/False column per reason code, from month-M information only."""
    sessions = rows["ga4_sessions_m"]
    return pd.DataFrame({
        "real_shortfall": score > 0,
        "rank_unstable": rows["position_volatility_m"] >= UNSTABLE_SPREAD,
        "client_wide_low_ctr": client_factor(rows) < CLIENT_LOW,
        "commercial_intent": rows["main_intent"].isin(BUYING),
        "low_engagement": (sessions >= MIN_SESSIONS) & (rows["ga4_engaged_sessions_m"] < ENGAGED_SHARE * sessions),
    }, index=rows.index)


ACTION_ORDER = [("rank_unstable", "Wait and re-check next month"), ("client_wide_low_ctr", "Review site-wide"),
                ("commercial_intent", "Check the search results page first"), ("low_engagement", "Improve the content")]
DEFAULT_ACTION = "Rewrite the title and description"


def choose_action(codes: pd.DataFrame) -> pd.Series:
    """Return the action of the first reason code that applies, in the fixed priority order."""
    return pd.Series(np.select([codes[code] for code, _ in ACTION_ORDER], [action for _, action in ACTION_ORDER], DEFAULT_ACTION),
                     index=codes.index)


show_table("The three candidate rules and what they read", pd.DataFrame({
    "Inputs": [", ".join(inputs) for inputs in RULE_INPUTS.values()],
    "Ranks by": ["missed clicks (Poisson-credible)", "missed clicks (short by the target's rule)", "impressions of flagged pages"],
}, index=pd.Index(list(RULES), name="Rule")))

**The three candidate rules and what they read**

| Rule | Inputs | Ranks by |
| --- | --- | --- |
| A | clicks_m, impressions_m, position_m, client_hash_id | missed clicks (Poisson-credible) |
| B | clicks_m, impressions_m, position_m, client_hash_id | missed clicks (short by the target's rule) |
| C | clicks_m, impressions_m, position_m | impressions of flagged pages |

**Finding:** all three rules read month M only. Rules A and B read the same four inputs (clicks, impressions, position, and the client, through the client factor); they differ only in how they judge the gap, five tiers with a Poisson test against position bins with the target's own rule. Rule C ignores the client entirely.


## 2. Build the ranked queue (writes the CSV)

### 2a. Which rule is hardest to beat?

**The freeze criterion, fixed before the results:** the rule with the highest mean Precision@50 across the three development pairs wins; if two rules are within 0.02 of each other, the one with the higher mean Capture@50 wins. Because the target depends heavily on page size (the leakage check), each rule is also reported on one fixed size band, pages with 1,000 to 9,999 impressions, and with the median size of its top 50.

**Method:** this cell scores every pair with all three rules, computes the metrics, and freezes the winner.


In [6]:
def precision_at_k(score: pd.Series, target: pd.Series, k: int) -> float:
    """Return the share of the top-k rows by score whose target is true."""
    return float(target.loc[score.sort_values(ascending=False, kind="stable").index[:k]].mean())


def capture_at_k(score: pd.Series, value: pd.Series, k: int) -> float:
    """Return the share of the total value (missed clicks on real shortfalls) held by the top-k rows."""
    return float(value.loc[score.sort_values(ascending=False, kind="stable").index[:k]].sum() / value.sum())


results = []
for name, rows in pairs.items():
    band = rows["impressions_m"].between(1_000, 9_999)
    for rule, scorer in RULES.items():
        score = scorer(rows)
        top = score.sort_values(ascending=False, kind="stable").index[:50]
        results.append({"Pair": name, "Rule": rule,
                        "Precision@50": precision_at_k(score, rows["target"], 50),
                        "Precision@20": precision_at_k(score, rows["target"], 20),
                        "Capture@50": capture_at_k(score, rows["missed_next"], 50),
                        "Precision@50, 1,000 to 9,999 impressions": precision_at_k(score[band], rows.loc[band, "target"], 50),
                        "Median impressions, top 50": rows.loc[top, "impressions_m"].median(),
                        "Base rate": rows["target"].mean()})
results = pd.DataFrame(results)
means = results.groupby("Rule")[["Precision@50", "Capture@50"]].mean()
# Rounded before comparing, so "within 0.02" is decided by the numbers themselves, not by floating-point noise.
best = round(means["Precision@50"].max(), 6)
contenders = means[means["Precision@50"].round(6) >= round(best - 0.02, 6)]
FROZEN = contenders["Capture@50"].idxmax()

show_table("Three candidate rules on the three development pairs", results.assign(
    **{column: results[column].map("{:.2f}".format) for column in ["Precision@50", "Precision@20", "Precision@50, 1,000 to 9,999 impressions"]},
    **{"Capture@50": results["Capture@50"].map("{:.1%}".format), "Base rate": results["Base rate"].map("{:.1%}".format),
       "Median impressions, top 50": results["Median impressions, top 50"].map("{:,.0f}".format)},
).set_index(["Pair", "Rule"]))
show_table("Means across the three pairs, and the frozen rule", pd.DataFrame({
    "Mean Precision@50": means["Precision@50"].map("{:.3f}".format),
    "Within 0.02 of the best": ["yes" if rule in contenders.index else "no" for rule in means.index],
    "Mean Capture@50": means["Capture@50"].map("{:.2%}".format),
    "Frozen": ["yes" if rule == FROZEN else "" for rule in means.index],
}).rename_axis("Rule"))

**Three candidate rules on the three development pairs**

| Pair | Rule | Precision@50 | Precision@20 | Capture@50 | Precision@50, 1,000 to 9,999 impressions | Median impressions, top 50 | Base rate |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 2026-02 → 2026-03 | A | 0.64 | 0.70 | 3.9% | 0.88 | 58,095 | 7.4% |
| 2026-02 → 2026-03 | B | 0.70 | 0.75 | 4.1% | 0.90 | 51,276 | 7.4% |
| 2026-02 → 2026-03 | C | 0.28 | 0.25 | 2.6% | 0.44 | 84,876 | 7.4% |
| 2026-03 → 2026-04 | A | 0.76 | 0.85 | 4.6% | 0.90 | 80,152 | 6.4% |
| 2026-03 → 2026-04 | B | 0.80 | 0.90 | 4.4% | 0.94 | 73,150 | 6.4% |
| 2026-03 → 2026-04 | C | 0.34 | 0.35 | 2.4% | 0.36 | 116,327 | 6.4% |
| 2026-04 → 2026-05 | A | 0.84 | 0.85 | 6.8% | 0.48 | 83,794 | 4.1% |
| 2026-04 → 2026-05 | B | 0.80 | 0.80 | 6.8% | 0.46 | 79,405 | 4.1% |
| 2026-04 → 2026-05 | C | 0.38 | 0.45 | 5.1% | 0.32 | 137,396 | 4.1% |

**Means across the three pairs, and the frozen rule**

| Rule | Mean Precision@50 | Within 0.02 of the best | Mean Capture@50 | Frozen |
| --- | ---: | --- | ---: | --- |
| A | 0.747 | yes | 5.09% |  |
| B | 0.767 | yes | 5.10% | yes |
| C | 0.333 | no | 3.35% |  |

**Finding:** rules A and B are effectively tied, and rule C is far behind. Mean Precision@50 is 0.767 for B (0.70, 0.80, 0.80), 0.747 for A (0.64, 0.76, 0.84), and 0.333 for C (0.28, 0.34, 0.38). B leads A by exactly 0.02, so by the criterion fixed in advance the two count as tied and Capture@50 decides: 5.10% for B against 5.09% for A. **Rule B is frozen as the baseline,** by a margin of one hundredth of a percentage point, so the honest reading is that both rules are about equally strong; B has the practical advantage of measuring a shortfall with the same yardstick as the target. On the fixed size band (1,000 to 9,999 impressions) the two stay close as well: 0.90, 0.94, and 0.46 for B against 0.88, 0.90, and 0.48 for A, with both dropping in the April to May pair, when that band's base rate is lowest. Rule C, the stand-in CTR flag, still beats random picking several times over, but its top 50 are simply the biggest flagged pages (median 84,876 to 137,396 impressions), whether or not they underperform for their position.


### 2b. The April 2026 queue

The queue ranks **every page visible in April 2026** with a valid position and a covered client, using only April's data, exactly as an editor would receive it on 1 May. Pages the contract would later drop (no impressions in May, or optimized by FlyRank during the pair) stay in the queue, because nobody knows that on 1 May; their May outcome is recorded instead.

**Method:** this cell builds April's pages from month-M data alone, checks that no May column is present, scores them with the frozen rule, attaches reason codes and an action, ranks every page with a real shortfall, records what happened in May, and writes the queue to `work/outputs/baseline_action_score.csv` (git ignores CSV files under `work/`, so the queue stays local; the tables here carry the results).


In [7]:
# April's pages from April's data alone; May is joined only afterwards, to record the outcome.
QUEUE_PAIR = ("2026-04", "2026-05")
month_m, month_next = QUEUE_PAIR
MONTH_COLUMNS = ["impressions", "clicks", "position_impressions", "sum_position", "sum_position_sq", "clicks_without_position",
                 "days_with_impressions", "ga4_sessions", "ga4_engaged_sessions"]
april = page_months[(page_months["month"] == month_m) & (page_months["position_impressions"] > 0)]
april = april[april["client_hash_id"].isin(covered_clients(month_m))]
queue = add_luck_test(add_month_measures(april.rename(columns={column: f"{column}_m" for column in MONTH_COLUMNS})
                                         .merge(content, on="content_hash_id", how="left").reset_index(drop=True), "m"), "m")
assert not [column for column in queue.columns if column.endswith("_next")], "A May column reached the queue before scoring."

queue["score"] = RULES[FROZEN](queue)
codes = reason_codes(queue, queue["score"])
queue = queue.join(codes)
queue["action"] = choose_action(codes)
queue["reason_codes"] = codes.apply(lambda row: ", ".join(code for code, applies in row.items() if applies), axis=1)
queue = queue[queue["score"] > 0].sort_values("score", ascending=False, kind="stable").reset_index(drop=True)
queue["rank"] = queue.index + 1

# What happened in May: the target where the page stayed eligible, otherwise why it dropped out.
pair_rows = pairs[f"{month_m} → {month_next}"].set_index("content_hash_id")
start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
in_pair = queue["content_hash_id"].isin(pair_rows.index)
ratio = pair_rows["clicks_next"] / pair_rows["expected_bins_next"]
queue["clicks_vs_expected_next"] = queue["content_hash_id"].map(ratio)
queue["impressions_next"] = queue["content_hash_id"].map(pair_rows["impressions_next"])
queue["outcome_next"] = np.select(
    [queue["content_hash_id"].isin(pair_rows.index[pair_rows["target"]]),
     in_pair & (queue["clicks_vs_expected_next"] <= 0.5), in_pair,
     pd.to_datetime(queue["last_optimized_date"]).between(start, end),
     ~queue["content_hash_id"].isin(page_months.loc[page_months["month"] == month_next, "content_hash_id"])],
    ["Real shortfall again", "Short again, but too little May traffic to rule out luck", "Recovered: over half of expected clicks",
     "Optimized by FlyRank during the pair", "No impressions in May"],
    "No valid position in May")

os.makedirs("work/outputs", exist_ok=True)
queue.rename(columns={"score": "missed_clicks_score", "expected_bins_m": "expected_clicks_m", "position_volatility_m": "position_spread_m"})[[
    "rank", "client_hash_id", "content_hash_id", "missed_clicks_score", "impressions_m", "clicks_m", "expected_clicks_m",
    "position_m", "position_spread_m", "main_intent", "content_type", "reason_codes", "action", "outcome_next",
]].to_csv("work/outputs/baseline_action_score.csv", index=False)

show("April 2026 queue", {
    "frozen rule": FROZEN,
    "pages visible in April (scored)": f"{len(april):,}",
    "pages in the queue (real shortfall in April)": f"{len(queue):,}",
    "written to": "work/outputs/baseline_action_score.csv",
})
show_table("Queue by action", queue["action"].value_counts().rename("Pages").map("{:,}".format).to_frame().rename_axis("Action"))

**April 2026 queue**

| Measure | Value |
| --- | --- |
| frozen rule | B |
| pages visible in April (scored) | 190,872 |
| pages in the queue (real shortfall in April) | 10,590 |
| written to | work/outputs/baseline_action_score.csv |

**Queue by action**

| Action | Pages |
| --- | ---: |
| Rewrite the title and description | 4,275 |
| Wait and re-check next month | 3,276 |
| Check the search results page first | 2,731 |
| Improve the content | 296 |
| Review site-wide | 12 |

**Finding:** the frozen rule scores all 190,872 pages visible in April 2026 and puts 10,590 of them (5.5%) in the queue, each with a real shortfall in April. The most common action is rewriting the title and description (4,275 pages), followed by waiting because the rank is still moving (3,276), checking the results page first on buying-intent keywords (2,731), improving the content where visitors leave quickly (296), and reviewing a whole site that clicks below normal (12).


## 3. Top-20 review

For each of the top 20: the action, the reason codes, a confidence note, what would make the pick wrong, and what actually happened in May. The data is pseudonymized, so the review reads the numbers, not the pages themselves.

**Confidence note, fixed in advance:** _high_ when the April shortfall is extremely unlikely to be luck (negative binomial p below 0.001), the rank is stable, and the client is not low across the board; _low_ when the rank is unstable or the page had under 1,000 impressions; _medium_ otherwise.

**Method:** this cell builds the review table for the top 20 pages of the queue.


In [8]:
# The top 20, one row each, with a confidence note and the most likely way the pick could be wrong.
WRONG_IF = {
    "Wait and re-check next month": "The rank is still moving; the shortfall may disappear once it settles",
    "Review site-wide": "The whole client clicks below normal (brand or page layout), so this page's snippet is not the cause",
    "Check the search results page first": "Ads and shopping results may take the clicks at this intent, which no rewrite fixes",
    "Improve the content": "Visitors leave quickly; the snippet may be fine and the page itself the problem",
    "Rewrite the title and description": "A results-page feature such as an AI answer or featured snippet may take the clicks",
}
top20 = queue.head(20)
confidence = np.select(
    [(top20["p_m"] < 0.001) & ~top20["rank_unstable"] & ~top20["client_wide_low_ctr"],
     top20["rank_unstable"] | (top20["impressions_m"] < 1_000)], ["High", "Low"], "Medium")
show_table("Top 20 of the April 2026 queue", pd.DataFrame({
    "Action": top20["action"].to_numpy(),
    "Reason codes": top20["reason_codes"].to_numpy(),
    "Missed clicks in April": top20["score"].map("{:,.0f}".format).to_numpy(),
    "Impressions": top20["impressions_m"].map("{:,.0f}".format).to_numpy(),
    "Position": top20["position_m"].map("{:.1f}".format).to_numpy(),
    "Confidence": confidence,
    "What would make it wrong": top20["action"].map(WRONG_IF).to_numpy(),
    "May outcome": top20["outcome_next"].to_numpy(),
    "May impressions": top20["impressions_next"].map(lambda value: f"{value:,.0f}" if pd.notna(value) else "n/a").to_numpy(),
    "May clicks ÷ expected": top20["clicks_vs_expected_next"].map(lambda value: f"{value:.2f}" if pd.notna(value) else "n/a").to_numpy(),
}, index=pd.Index(top20["rank"], name="Rank")))

**Top 20 of the April 2026 queue**

| Rank | Action | Reason codes | Missed clicks in April | Impressions | Position | Confidence | What would make it wrong | May outcome | May impressions | May clicks ÷ expected |
| ---: | --- | --- | ---: | ---: | ---: | --- | --- | --- | --- | --- |
| 1 | Rewrite the title and description | real_shortfall | 858 | 144,986 | 2.2 | High | A results-page feature such as an AI answer or featured snippet may take the clicks | Short again, but too little May traffic to rule out luck | 103 | 0.00 |
| 2 | Improve the content | real_shortfall, low_engagement | 746 | 305,321 | 2.4 | Medium | Visitors leave quickly; the snippet may be fine and the page itself the problem | Optimized by FlyRank during the pair | n/a | n/a |
| 3 | Check the search results page first | real_shortfall, commercial_intent | 598 | 186,872 | 9.4 | High | Ads and shopping results may take the clicks at this intent, which no rewrite fixes | Optimized by FlyRank during the pair | n/a | n/a |
| 4 | Improve the content | real_shortfall, low_engagement | 587 | 227,139 | 4.0 | High | Visitors leave quickly; the snippet may be fine and the page itself the problem | Real shortfall again | 176,759 | 0.29 |
| 5 | Check the search results page first | real_shortfall, commercial_intent | 549 | 138,322 | 8.2 | High | Ads and shopping results may take the clicks at this intent, which no rewrite fixes | Real shortfall again | 88,175 | 0.00 |
| 6 | Improve the content | real_shortfall, low_engagement | 548 | 191,648 | 2.4 | High | Visitors leave quickly; the snippet may be fine and the page itself the problem | Optimized by FlyRank during the pair | n/a | n/a |
| 7 | Check the search results page first | real_shortfall, commercial_intent, low_engagement | 434 | 150,907 | 3.8 | High | Ads and shopping results may take the clicks at this intent, which no rewrite fixes | Real shortfall again | 151,562 | 0.28 |
| 8 | Check the search results page first | real_shortfall, commercial_intent, low_engagement | 431 | 228,149 | 4.7 | High | Ads and shopping results may take the clicks at this intent, which no rewrite fixes | Real shortfall again | 155,348 | 0.37 |
| 9 | Rewrite the title and description | real_shortfall | 411 | 90,385 | 2.3 | High | A results-page feature such as an AI answer or featured snippet may take the clicks | Real shortfall again | 21,814 | 0.11 |
| 10 | Check the search results page first | real_shortfall, commercial_intent, low_engagement | 383 | 152,152 | 5.1 | High | Ads and shopping results may take the clicks at this intent, which no rewrite fixes | Real shortfall again | 235,817 | 0.07 |
| 11 | Check the search results page first | real_shortfall, commercial_intent | 309 | 109,923 | 2.3 | High | Ads and shopping results may take the clicks at this intent, which no rewrite fixes | Recovered: over half of expected clicks | 11,709 | 0.92 |
| 12 | Rewrite the title and description | real_shortfall | 303 | 80,426 | 15.1 | High | A results-page feature such as an AI answer or featured snippet may take the clicks | Real shortfall again | 61,899 | 0.03 |
| 13 | Review site-wide | real_shortfall, client_wide_low_ctr, commercial_intent | 281 | 267,960 | 10.2 | Medium | The whole client clicks below normal (brand or page layout), so this page's snippet is not the cause | Real shortfall again | 16,123 | 0.00 |
| 14 | Improve the content | real_shortfall, low_engagement | 277 | 96,024 | 4.9 | High | Visitors leave quickly; the snippet may be fine and the page itself the problem | Real shortfall again | 98,383 | 0.28 |
| 15 | Check the search results page first | real_shortfall, commercial_intent | 276 | 128,092 | 6.4 | High | Ads and shopping results may take the clicks at this intent, which no rewrite fixes | Real shortfall again | 17,055 | 0.00 |
| 16 | Check the search results page first | real_shortfall, commercial_intent, low_engagement | 262 | 91,263 | 4.1 | High | Ads and shopping results may take the clicks at this intent, which no rewrite fixes | Recovered: over half of expected clicks | 75,011 | 0.51 |
| 17 | Wait and re-check next month | real_shortfall, rank_unstable | 248 | 43,060 | 5.0 | Low | The rank is still moving; the shortfall may disappear once it settles | Real shortfall again | 5,043 | 0.31 |
| 18 | Rewrite the title and description | real_shortfall | 237 | 91,909 | 6.2 | High | A results-page feature such as an AI answer or featured snippet may take the clicks | Optimized by FlyRank during the pair | n/a | n/a |
| 19 | Wait and re-check next month | real_shortfall, rank_unstable | 237 | 109,367 | 25.1 | Low | The rank is still moving; the shortfall may disappear once it settles | Real shortfall again | 13,479 | 0.13 |
| 20 | Rewrite the title and description | real_shortfall | 226 | 56,908 | 3.2 | High | A results-page feature such as an AI answer or featured snippet may take the clicks | Recovered: over half of expected clicks | 20,844 | 0.59 |

**Finding:** 12 of the top 20 fell short again in May. Four more (ranks 2, 3, 6, and 18) were optimized by FlyRank itself during the pair, so FlyRank's own team chose to work on a fifth of this list. Three recovered to over half of their expected clicks (ranks 11, 16, and 20). The first-ranked page is a different problem altogether: it fell from 144,986 impressions in April to 103 in May, so it lost its search visibility rather than its click-through, which this queue is not built to catch. Of the 16 picks that stayed measurable and were not optimized, 12 (75%) were real shortfalls again. The confidence note did not yet separate hits from misses: both low-confidence picks (ranks 17 and 19, unstable rank) fell short again, while all four picks that did not were marked high.


## 4. Weak picks + leakage check

**Method:** this cell lists the top-20 picks that did not fall short again in May and summarizes how the whole top 20 and top 50 fared.


In [9]:
# Picks that did not hold up in May, and the top 20 and top 50 outcomes overall.
weak = top20[top20["outcome_next"] != "Real shortfall again"]
show_table("Top-20 picks that were not a real shortfall again in May", pd.DataFrame({
    "May outcome": weak["outcome_next"].to_numpy(),
    "April impressions": weak["impressions_m"].map("{:,.0f}".format).to_numpy(),
    "May impressions": weak["impressions_next"].map(lambda value: f"{value:,.0f}" if pd.notna(value) else "n/a").to_numpy(),
    "May clicks ÷ expected": weak["clicks_vs_expected_next"].map(lambda value: f"{value:.2f}" if pd.notna(value) else "n/a").to_numpy(),
    "Reason codes": weak["reason_codes"].to_numpy(),
    "Position spread in April": weak["position_volatility_m"].map("{:.1f}".format).to_numpy(),
}, index=pd.Index(weak["rank"], name="Rank")))
outcomes = pd.DataFrame({size: queue.head(size)["outcome_next"].value_counts() for size in (20, 50)}).fillna(0).astype(int)
show_table("May outcomes of the queue's top 20 and top 50", outcomes.rename(columns={20: "Top 20", 50: "Top 50"}).rename_axis("May outcome"))

**Top-20 picks that were not a real shortfall again in May**

| Rank | May outcome | April impressions | May impressions | May clicks ÷ expected | Reason codes | Position spread in April |
| ---: | --- | ---: | --- | --- | --- | ---: |
| 1 | Short again, but too little May traffic to rule out luck | 144,986 | 103 | 0.00 | real_shortfall | 0.3 |
| 2 | Optimized by FlyRank during the pair | 305,321 | n/a | n/a | real_shortfall, low_engagement | 0.3 |
| 3 | Optimized by FlyRank during the pair | 186,872 | n/a | n/a | real_shortfall, commercial_intent | 0.3 |
| 6 | Optimized by FlyRank during the pair | 191,648 | n/a | n/a | real_shortfall, low_engagement | 0.5 |
| 11 | Recovered: over half of expected clicks | 109,923 | 11,709 | 0.92 | real_shortfall, commercial_intent | 0.8 |
| 16 | Recovered: over half of expected clicks | 91,263 | 75,011 | 0.51 | real_shortfall, commercial_intent, low_engagement | 0.8 |
| 18 | Optimized by FlyRank during the pair | 91,909 | n/a | n/a | real_shortfall | 1.1 |
| 20 | Recovered: over half of expected clicks | 56,908 | 20,844 | 0.59 | real_shortfall | 2.8 |

**May outcomes of the queue's top 20 and top 50**

| May outcome | Top 20 | Top 50 |
| --- | ---: | ---: |
| Optimized by FlyRank during the pair | 4 | 7 |
| Real shortfall again | 12 | 34 |
| Recovered: over half of expected clicks | 3 | 8 |
| Short again, but too little May traffic to rule out luck | 1 | 1 |

**Finding:** across the top 50, 34 pages fell short again in May, 7 were optimized by FlyRank during the pair, 8 recovered, and 1 lost almost all its traffic; of the 43 pages FlyRank did not touch, 34 (79%) were real shortfalls again. Two of the three recoveries in the top 20 target buying-intent keywords, which matches the signal audit's finding that buying intent clicks at least as well as informational intent, so the `commercial_intent` code may be pointing at pages that recover on their own. The other recovery (rank 20) had the largest daily position spread among the misses, 2.8 places.


**Method:** this cell checks, in code, that neither the frozen rule nor the reason codes read anything from May or any FlyRank product field. The queue was also checked above to hold no May column at the moment it was scored.


In [10]:
# Inputs of the frozen rule and of the reason codes: month M only, no product decisions.
PRODUCT_FIELDS = {"last_optimized_date", "optimization_eligible_date", "is_deleted", "is_published", "content_updated_date"}
for inputs in (RULE_INPUTS[FROZEN], CODE_INPUTS):
    assert not [column for column in inputs if column.endswith("_next")], "A month M+1 field feeds the score or a reason code."
    assert not set(inputs) & PRODUCT_FIELDS, "A product field feeds the score or a reason code."
show("Leakage check (all passed)", {
    "frozen rule reads": ", ".join(RULE_INPUTS[FROZEN]),
    "reason codes read": ", ".join(CODE_INPUTS),
    "month M+1 fields used for scoring": "none (May joined only after ranking, to record outcomes)",
    "product fields used for scoring": "none (the optimization date only labels the May outcome)",
})

**Leakage check (all passed)**

| Measure | Value |
| --- | --- |
| frozen rule reads | clicks_m, impressions_m, position_m, client_hash_id |
| reason codes read | position_volatility_m, clicks_m, impressions_m, position_m, main_intent, ga4_sessions_m, ga4_engaged_sessions_m |
| month M+1 fields used for scoring | none (May joined only after ranking, to record outcomes) |
| product fields used for scoring | none (the optimization date only labels the May outcome) |

**Finding:** the leakage check passes: the frozen rule and every reason code read month-M fields only, May is joined after the ranking is final, and FlyRank's optimization date only labels the May outcome; it never touches a score.


## Limitations

- **The queue is checked against one month.** April's picks are judged on May only; the frozen rule's general strength comes from the three-pair comparison, not from this one queue.
- **Reasons and actions are rules of thumb.** The thresholds (5 places, 30 sessions, half engaged, half the expected clicks) are readable starting points chosen in advance, not tuned values.
- **No page is seen directly.** The data is pseudonymized, so titles, snippets, and results pages cannot be checked; "what would make it wrong" names the most likely alternative explanation, not a verified one.
- **Size still matters.** Precision@50 rewards large pages; the fixed size band and the median size of each top 50 are reported so a later model can be compared on the same footing.
- **The frozen rule stays frozen.** Any later model is compared against it unchanged, on the same pairs and metrics.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
